# ENGR 1451/2451 — Weekly Assignment 7
**10 points · 4 graded parts**  
**Due:** See Canvas.

Practice confounding, standard error, and confidence intervals. Show your Python code and keep written answers brief.


## What to hand in

Work with the **course tutor** in the class project. Use it as your AI help; it keeps the record, so no separate log is needed. Your code and written answers must still be your own.

At the end of each working session, type **"make my record"** in the chat. Copy its block into **Session records** below, placing each new record after the previous one.

To continue later, start a new chat in the same project and state the assignment and where you stopped. Session records are not graded for correctness or how much help you needed.


## Setup

All data are embedded; no additional files are needed. Keep full precision in calculations and round only displayed results.


In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats

pd.set_option("display.precision", 4)


## LE7-1. Group composition and confounding (4 pts)

These synthetic inspection records compare two manufacturing methods. Methods were **not randomly assigned** to parts.

Calculate pass rates for each method within each part type and overall. For overall rates, use `groupby()` and sum the counts before dividing; do not average percentages.

Which method has the higher overall rate? Which has the higher rate within each part type? In two sentences, explain the reversal using group composition and why these observations do not establish a causal effect of method.

**Answer:** B has the higher pass rate overall, A had a higher rate for each part type individually though. This reversal happens due to the varying group sizes for each part type between methods. Though A technically passed less parts overall, its individual rates for simple and complex were far better, but since its higher pass rate was its less numerous part type, and its worse rate was on the higher part count, it seems that it has an overall worse pass rate. These do not establish a causal effect due to method, as the sample sizes for complex and simple parts were different between methods.


In [2]:
inspection = pd.DataFrame({
    "method": ["A", "A", "B", "B"],
    "part_type": ["Simple", "Complex", "Simple", "Complex"],
    "inspected": [20, 80, 80, 20],
    "passed": [18, 48, 64, 10],
})

# Calculate and display pass rates within each part type.
# Group by method, sum inspected/passed counts, and calculate overall rates.
by_part = inspection.groupby("method")[["inspected", "passed"]].sum()
passrate_a, passrate_b = by_part["passed"]["A"]/by_part["inspected"]["A"], by_part["passed"]["B"]/by_part["inspected"]["B"]
print(f"Passrate method A: {passrate_a:.4f}")
print(f"Passrate method B: {passrate_b:.4f}")

Passrate method A: 0.6600
Passrate method B: 0.7400


## LE7-2. Estimate a population mean (4 pts)

The synthetic tensile strengths below are in **MPa**. Assume an independent random sample from a stable, approximately normal production process. Its population mean and SD are unknown.

### A. Estimate the SD and SE (2 pts)

Display $n$, $\bar x$, sample SD $s$ using `std(ddof=1)`, and estimated SE using `stats.sem(..., ddof=1)`.

In one sentence, distinguish what $s$ and $s/\sqrt{n}$ measure.

**Answer:** SD is better for the spread of individual points, whereas s/sqrt(n) is standard error, and relates population means and their spread.


In [3]:
strength = np.array([
    48.2, 51.4, 49.7, 52.1, 46.8, 50.5, 53.0, 47.6,
    49.1, 51.8, 48.9, 50.7, 47.9, 52.5, 49.8, 50.2,
])

# Calculate and display n, sample_mean, sample_sd, and estimated_se.
n = len(strength)
sample_mean = strength.mean()
sample_sd = strength.std(ddof=1)
estimated_se = sample_sd / np.sqrt(n)

print(f"N = {n}")
print(f"Sample Mean = {np.round(sample_mean, 4 )}")
print(f"Sample SD = {np.round(sample_sd, 4 )}")
print(f"Estimated SD = {np.round(estimated_se, 4 )}")

N = 16
Sample Mean = 50.0125
Sample SD = 1.8518
Estimated SD = 0.463


### B. Construct and interpret a confidence interval (2 pts)

Because $\sigma$ is unknown, use a **Student's $t$ interval**:

$$\bar x \pm t^*\frac{s}{\sqrt{n}},\qquad df=n-1.$$

Calculate `t_star = stats.t.ppf(0.975, df=n-1)` for 95% confidence. Display $df$, $t^*$, and both interval endpoints in MPa.

Interpret the interval for the **population mean**. Does it describe where 95% of individual strengths fall? Would a 99% interval using the same sample be wider or narrower? No second calculation is required.

**Answer:** The interval shows how the range that 95% of population means would likely fall into for this dataset. A 99% interval would need to be larger, as it would have to cover another 4% of population means outside of the 95% covered by the previous interval


In [4]:
# Calculate the t critical value and margin of error.
t_star = stats.t.ppf(.975, df=n-1)

# Display df, t_star, and the lower/upper confidence limits in MPa.
upper_confidence_interval = sample_mean + t_star*estimated_se
lower_confidence_interval = sample_mean - t_star*estimated_se

print(f"df = {n-1}")
print(f"t* = {t_star:.4f}")
print(f"The upper and lower bounds of the confidence interval fall between {upper_confidence_interval:.4f} and {lower_confidence_interval:.4f} MPa respectively")

df = 15
t* = 2.1314
The upper and lower bounds of the confidence interval fall between 50.9993 and 49.0257 MPa respectively


## LE7-3. Estimate a population percentage (2 pts)

*Adapted from Chapter 21, Review Exercise 2 (printed p. 391; PDF p. 17).*

In the chapter's historical survey, a simple random sample of **500 households** from a town of **25,000 households** included **239 with internet access**.

Use Python to calculate the estimated proportion and its estimated SE:

$$\hat p=\frac{x}{n},\qquad
\widehat{\operatorname{SE}}(\hat p)=\sqrt{\frac{\hat p(1-\hat p)}{n}}.$$

For this large-sample proportion, use the normal approximation with `z_star = stats.norm.ppf(0.975)`:

$$\hat p\pm z^*\widehat{\operatorname{SE}}(\hat p).$$

Ignore the finite-population correction; the sample includes only 2% of the town's households.

Display the estimated **percentage**, its SE in **percentage points**, and the approximate **95% confidence interval as percentages**. Interpret the interval for all households in the town in one sentence.

**Answer:**
The interval shows that 95% of the time the percentage of houses within a town with internet is between 43.2% and 52.18%.

In [5]:
total_households = 25_000
sampled_households = 500
with_internet = 239

# Calculate p_hat and its estimated SE using np.sqrt().
p_hat = with_internet/sampled_households
estimated_se = np.sqrt((p_hat*(1-p_hat))/sampled_households)

# Find z_star and calculate the lower and upper confidence limits.
z_star = stats.norm.ppf(.975)
upper_lim, lower_lim = p_hat+z_star*estimated_se, p_hat-z_star*estimated_se

# Display the estimate, SE, and interval in percentage units.
print(f"Estimated percentage:{p_hat*100}%")
print(f"SE: {estimated_se*100:.2f}%")
print(f"95% confidence interval from {lower_lim*100:.2f}% to {upper_lim*100:.2f}%")

Estimated percentage:47.8%
SE: 2.23%
95% confidence interval from 43.42% to 52.18%


## Session records

Paste the block from **"make my record"** after each working session. Keep all session records in order.


<!-- Paste your session record blocks below, one after another. -->

*(session records go here)*


## Before you submit

1. **Restart Kernel and Run All Cells**; confirm there are no errors.
2. Check that calculations, tables, and written answers are visible.
3. Include your session record block(s).
4. Save and submit the completed `.ipynb` file in Canvas.


In [ ]:
{
  "schema_version": "engr-tutor-export-2",
  "assignment": "WA7",
  "export_n": 1,
  "session_date": "2026-10-20",
  "messages": [
    {"i": 1, "who": "student", "text": "could you help me with le7-1"},
    {"i": 2, "who": "tutor", "text": "Happy to help with LE7-1. Starting with the problem that has two clear calculation steps is a good place to begin.\n\nThe problem needs two displays. First, a pass rate for each of the four method-and-part-type rows. Second, an overall rate for each method, built from its summed counts. Each row of `inspection` already holds one method within one part type, so its rate is a single division.\n\nFor the overall rates, which column goes inside `groupby()`, and what has to happen to the counts before you divide?\n\nIf you've already written something in the cell, paste it here with its output or any error, and I'll start from what you have."},
    {"i": 3, "who": "student", "text": "\"part_type\""},
    {"i": 4, "who": "tutor", "text": "Good thinking about the grouping key. That's the right thing to decide first.\n\nGrouping by `part_type` would give one overall rate per part type, with both methods combined. The prompt asks for an overall rate for each *method*. The comment in the starter cell spells it out: \"Group by method, sum inspected/passed counts, and calculate overall rates.\"\n\nHere's what `groupby()` does in general. It splits the rows into groups, one group for each distinct value in the key column. The aggregation you apply then runs once per group. For example, with a frame of clinic counts:\n\n```python\nclinic.groupby(\"ward\")[[\"patients\", \"recovered\"]].sum()\n```\n\nThis gives one row per ward. Each row holds that ward's total patients and total recovered, added up across every row belonging to that ward. You'd divide those totals afterward to get each ward's overall rate.\n\nSo for LE7-1, which column should be the key so that you end up with one row for A and one row for B?"},
    {"i": 5, "who": "student", "text": "[\"inspected\", \"passed\"]"},
    {"i": 6, "who": "tutor", "text": "You've got the right columns. `inspected` and `passed` are the counts you want to add up within each group.\n\nThere are two separate pieces in a grouped sum, and they go in different places:\n\n```python\nframe.groupby(KEY)[COLUMNS_TO_SUM].sum()\n```\n\nThe key goes inside `groupby()`. It decides which rows belong together, so you get one output row per distinct value of that column. The columns to sum go in the selection after it. That's what you just named.\n\nOne syntax detail: when you select more than one column after `groupby()`, the list needs its own brackets inside the selection brackets, like `[[\"a\", \"b\"]]`. With single brackets, `[\"a\", \"b\"]` raises a `ValueError` about subsetting with a tuple.\n\nThat leaves the key. In `inspection`, which column holds the values \"A\" and \"B\"?"},
    {"i": 7, "who": "student", "text": "method"},
    {"i": 8, "who": "tutor", "text": "That's it. `method` is the key, so you'll get one row for A and one for B, each holding that method's summed counts.\n\nYou now have every piece for the overall rates: the key, the two count columns in double brackets, `.sum()`, and then a division of the summed passed column by the summed inspected column. Store the grouped result in a variable first, then divide its columns, the same way you'd divide two columns of any DataFrame.\n\nDon't forget the other display, the within-type rates. That one doesn't need grouping at all, since each row of `inspection` is already one method within one part type. It's just passed divided by inspected, row by row.\n\nOne thing to watch: if you add a rate column to `inspection` and later run a grouped `.sum()` on the whole frame, pandas will add up the rates too. That summed column isn't a rate. Selecting just the two count columns, as you planned, avoids that.\n\nGive the cell a try and paste it here with its output. Which of the two displays do you want to write first?"}
  ],
  "events": [
    {"i": 4, "part": "7-1", "type": "explain"},
    {"i": 6, "part": "7-1", "type": "explain"}
  ],
  "parts": [{"part": "7-1", "exchanges": 4, "max_tier": 2, "passed": false}],
  "tutor_assessment": [
    "The longest stall was on LE7-1 before any code was written: choosing the groupby() key. The student first named part_type, then named the columns to sum rather than the key, and reached method after the key and the column selection were explained as separate pieces of a grouped sum.",
    "No unstuck or release-valve use occurred this session."
  ]
}